# Weight Initialization in TensorFlow / Keras

This notebook mirrors the PyTorch lab in TensorFlow/Keras, with additional focus on the `VarianceScaling` abstraction and Keras's `kernel_initializer` API. You will:
1. **Apply all `tf.keras.initializers`** and verify the resulting statistics
2. **Reconstruct GlorotUniform, HeNormal, and LecunNormal** using `VarianceScaling` from first principles
3. **Verify PyTorch/TensorFlow parity** — compare histograms and variance for matched initialiser pairs
4. **Train on CIFAR-10** with Tanh+Xavier, ReLU+Xavier, and ReLU+He — compare training curves and activation statistics
5. **Use the `kernel_initializer` API** to build and inspect a ResNet-style block
6. **Audit Keras layer defaults** for `Dense`, `Conv2D`, `LSTM`, and `BatchNormalization`

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, initializers

tf.random.set_seed(42)
np.random.seed(42)

print(f'TensorFlow {tf.__version__}')
print(f'GPUs available: {len(tf.config.list_physical_devices("GPU"))}')

def check_stats(name, arr, expected_mean=0.0, expected_var=None, atol_mean=0.05, atol_var=0.1):
    m = float(np.mean(arr))
    v = float(np.var(arr))
    mean_ok = abs(m - expected_mean) < atol_mean
    var_ok  = True if expected_var is None else abs(v - expected_var) < atol_var
    status  = '\u2713' if (mean_ok and var_ok) else '\u2717'
    var_str = f'var={v:.5f} (expected~{expected_var:.5f})' if expected_var is not None else f'var={v:.5f}'
    print(f'{status} {name:<44} mean={m:.4f}  {var_str}')

## Section 1 — All `tf.keras.initializers` Classes

We instantiate every built-in initialiser, generate a `(512, 512)` tensor, and verify the resulting mean, variance, and structural properties.

In [ ]:
shape = (512, 512)

# --- Constant initializers ---
print('=== Constant initializers ===')
t = initializers.Zeros()(shape=shape).numpy()
check_stats('Zeros',          t, expected_mean=0.0, expected_var=0.0)

t = initializers.Ones()(shape=shape).numpy()
check_stats('Ones',           t, expected_mean=1.0, expected_var=0.0)

t = initializers.Constant(value=3.14)(shape=shape).numpy()
check_stats('Constant(3.14)', t, expected_mean=3.14, expected_var=0.0)

t = initializers.Identity()(shape=(256, 256)).numpy()
identity_ok = np.allclose(t, np.eye(256))
print(f"{'\u2713' if identity_ok else '\u2717'} Identity  diagonal=1, off-diagonal=0: {identity_ok}")

# --- Random initializers ---
print('\n=== Random initializers ===')
t = initializers.RandomUniform(minval=-1.0, maxval=1.0)(shape=shape).numpy()
check_stats('RandomUniform(-1, 1)',      t, expected_mean=0.0, expected_var=1/3)

t = initializers.RandomNormal(mean=0.0, stddev=1.0)(shape=shape).numpy()
check_stats('RandomNormal(0, 1)',        t, expected_mean=0.0, expected_var=1.0)

t = initializers.TruncatedNormal(mean=0.0, stddev=1.0)(shape=shape).numpy()
out_of_bounds = (np.abs(t) > 2.0).sum()
print(f'\u2713 TruncatedNormal  values |x|>2: {out_of_bounds} (should be 0)')
check_stats('TruncatedNormal(0, 1)',     t, expected_mean=0.0)

# --- Variance-scaling initializers ---
print('\n=== Variance-scaling initializers ===')
fan_in = fan_out = 512

t = initializers.GlorotUniform()(shape=shape).numpy()
glorot_var = 2.0 / (fan_in + fan_out)  # 2 / (fan_in + fan_out)
check_stats('GlorotUniform (Xavier)',    t, expected_mean=0.0, expected_var=glorot_var)

t = initializers.GlorotNormal()(shape=shape).numpy()
check_stats('GlorotNormal (Xavier)',     t, expected_mean=0.0, expected_var=glorot_var)

t = initializers.HeUniform()(shape=shape).numpy()
he_var = 2.0 / fan_in
check_stats('HeUniform (Kaiming)',       t, expected_mean=0.0, expected_var=he_var)

t = initializers.HeNormal()(shape=shape).numpy()
check_stats('HeNormal (Kaiming)',        t, expected_mean=0.0, expected_var=he_var)

t = initializers.LecunUniform()(shape=shape).numpy()
lecun_var = 1.0 / fan_in
check_stats('LecunUniform',             t, expected_mean=0.0, expected_var=lecun_var)

t = initializers.LecunNormal()(shape=shape).numpy()
check_stats('LecunNormal',              t, expected_mean=0.0, expected_var=lecun_var)

# --- Orthogonal ---
print('\n=== Orthogonal ===')
t = initializers.Orthogonal(gain=1.0)(shape=(256, 256)).numpy()
diff = np.abs(t.T @ t - np.eye(256)).max()
print(f'\u2713 Orthogonal  max|W^T W - I| = {diff:.2e}')

## Section 2 — `VarianceScaling` Deep-Dive

`tf.keras.initializers.VarianceScaling` is the single general initialiser underlying GlorotUniform, HeNormal, and LecunNormal. Its three parameters:
- **`scale`**: multiplied into the variance
- **`mode`**: `fan_in`, `fan_out`, or `fan_avg` — which fan value divides scale
- **`distribution`**: `uniform`, `truncated_normal`, or `untruncated_normal`

We reconstruct the three named initialisers from scratch and verify they match.

In [ ]:
shape = (1024, 1024)
fan_in = fan_out = 1024

# GlorotUniform = VarianceScaling(scale=1, mode='fan_avg', distribution='uniform')
# Var = scale / fan_avg = 1 / ((fan_in+fan_out)/2) * correction_for_uniform
# Keras uses limit = sqrt(3 * scale / fan_avg), U(-limit, limit)
glorot_vs = initializers.VarianceScaling(
    scale=1.0, mode='fan_avg', distribution='uniform')(shape=shape).numpy()
glorot_named = initializers.GlorotUniform()(shape=shape).numpy()
expected_glorot_var = 2.0 / (fan_in + fan_out)
print('GlorotUniform reconstruction:')
check_stats('  VarianceScaling(1, fan_avg, uniform)', glorot_vs,    expected_var=expected_glorot_var)
check_stats('  GlorotUniform()', glorot_named, expected_var=expected_glorot_var)
print(f'  Max abs diff between them: {np.abs(np.var(glorot_vs) - np.var(glorot_named)):.2e}')

print()

# HeNormal = VarianceScaling(scale=2, mode='fan_in', distribution='truncated_normal')
# Std = sqrt(2 / fan_in), truncated at 2 std
he_vs = initializers.VarianceScaling(
    scale=2.0, mode='fan_in', distribution='truncated_normal')(shape=shape).numpy()
he_named = initializers.HeNormal()(shape=shape).numpy()
# Truncated normal has variance ~0.8798 * sigma^2 when truncated at +/-2sigma
expected_he_var = 2.0 / fan_in
print('HeNormal reconstruction:')
check_stats('  VarianceScaling(2, fan_in, trunc_normal)', he_vs,    expected_var=expected_he_var, atol_var=0.01)
check_stats('  HeNormal()', he_named, expected_var=expected_he_var, atol_var=0.01)
print(f'  Max abs diff between vars:  {abs(np.var(he_vs) - np.var(he_named)):.2e}')

print()

# LecunNormal = VarianceScaling(scale=1, mode='fan_in', distribution='truncated_normal')
lecun_vs = initializers.VarianceScaling(
    scale=1.0, mode='fan_in', distribution='truncated_normal')(shape=shape).numpy()
lecun_named = initializers.LecunNormal()(shape=shape).numpy()
expected_lecun_var = 1.0 / fan_in
print('LecunNormal reconstruction:')
check_stats('  VarianceScaling(1, fan_in, trunc_normal)', lecun_vs,    expected_var=expected_lecun_var, atol_var=0.005)
check_stats('  LecunNormal()', lecun_named, expected_var=expected_lecun_var, atol_var=0.005)

print()
print('Summary of VarianceScaling parameters:')
print(f'  {"Initializer":<30} {"scale":<8} {"mode":<12} {"distribution"}')
print(f'  {"-"*60}')
print(f'  {"GlorotUniform":<30} {"1.0":<8} {"fan_avg":<12} {"uniform"}')
print(f'  {"GlorotNormal":<30} {"1.0":<8} {"fan_avg":<12} {"truncated_normal"}')
print(f'  {"HeUniform":<30} {"2.0":<8} {"fan_in":<12} {"uniform"}')
print(f'  {"HeNormal":<30} {"2.0":<8} {"fan_in":<12} {"truncated_normal"}')
print(f'  {"LecunUniform":<30} {"1.0":<8} {"fan_in":<12} {"uniform"}')
print(f'  {"LecunNormal":<30} {"1.0":<8} {"fan_in":<12} {"truncated_normal"}')

## Section 3 — PyTorch / TensorFlow Parity Check

For each matched initialiser pair we generate 50,000 samples from each framework and compare histograms and empirical variance.

In [ ]:
import torch
import torch.nn as nn

N = 50000
fan_in_p = fan_in

pairs = [
    (
        'GlorotUniform / xavier_uniform_',
        initializers.GlorotUniform()(shape=(N,)).numpy(),
        nn.init.xavier_uniform_(torch.empty(N, 1)).flatten().numpy(),
    ),
    (
        'HeNormal / kaiming_normal_',
        initializers.HeNormal()(shape=(N,)).numpy(),
        nn.init.kaiming_normal_(torch.empty(N, 1), nonlinearity='relu').flatten().numpy(),
    ),
    (
        'LecunNormal / xavier_normal_ (scaled)',
        initializers.LecunNormal()(shape=(N,)).numpy(),
        # LeCun = N(0, 1/fan_in); no direct PyTorch equivalent — use normal_ with matching std
        nn.init.normal_(torch.empty(N), std=math.sqrt(1.0 / fan_in_p)).numpy(),
    ),
    (
        'Orthogonal / orthogonal_',
        initializers.Orthogonal()(shape=(256, 256)).numpy().flatten(),
        nn.init.orthogonal_(torch.empty(256, 256)).flatten().numpy(),
    ),
]

fig, axes = plt.subplots(1, len(pairs), figsize=(18, 4))
for ax, (name, tf_samples, pt_samples) in zip(axes, pairs):
    ax.hist(tf_samples, bins=80, alpha=0.6, density=True, label='TensorFlow', color='steelblue')
    ax.hist(pt_samples, bins=80, alpha=0.6, density=True, label='PyTorch',    color='darkorange')
    ax.set_title(name, fontsize=8)
    ax.set_xlabel('Value'); ax.set_ylabel('Density')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

plt.suptitle('TensorFlow vs PyTorch Initialiser Parity', y=1.02)
plt.tight_layout(); plt.show()

print(f'\n{"Initialiser pair":<40} {"TF var":<12} {"PT var":<12} {"Abs diff"}')
print('-' * 72)
for name, tf_s, pt_s in pairs:
    tv, pv = np.var(tf_s), np.var(pt_s)
    print(f'{name:<40} {tv:<12.6f} {pv:<12.6f} {abs(tv-pv):.2e}')

## Section 4 — Initialization on CIFAR-10

We train a small convolutional network under three conditions and compare training loss, validation accuracy, and per-layer activation variance at epoch 1:
- **(a)** Tanh + GlorotUniform
- **(b)** ReLU + GlorotUniform
- **(c)** ReLU + HeNormal

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

mean = x_train.mean(axis=(0, 1, 2), keepdims=True)
std  = x_train.std(axis=(0, 1, 2),  keepdims=True)
x_train = (x_train - mean) / (std + 1e-7)
x_test  = (x_test  - mean) / (std + 1e-7)

print(f'CIFAR-10: x_train={x_train.shape}  x_test={x_test.shape}')

In [ ]:
def build_conv_net(activation, kernel_init):
    """Small 3-block Conv net with consistent architecture."""
    model = keras.Sequential([
        layers.Conv2D(32, 3, padding='same', activation=activation,
                      kernel_initializer=kernel_init, input_shape=(32, 32, 3)),
        layers.Conv2D(32, 3, padding='same', activation=activation,
                      kernel_initializer=kernel_init),
        layers.MaxPooling2D(2),

        layers.Conv2D(64, 3, padding='same', activation=activation,
                      kernel_initializer=kernel_init),
        layers.Conv2D(64, 3, padding='same', activation=activation,
                      kernel_initializer=kernel_init),
        layers.MaxPooling2D(2),

        layers.Flatten(),
        layers.Dense(256, activation=activation, kernel_initializer=kernel_init),
        layers.Dense(10),
    ])
    return model


configs = [
    ('Tanh + GlorotUniform',  'tanh',  'glorot_uniform'),
    ('ReLU + GlorotUniform',  'relu',  'glorot_uniform'),
    ('ReLU + HeNormal',       'relu',  'he_normal'),
]

histories = {}

for label, act, init in configs:
    print(f'\nTraining: {label}')
    tf.random.set_seed(42)
    model = build_conv_net(act, init)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )
    hist = model.fit(
        x_train, y_train, batch_size=256, epochs=10,
        validation_data=(x_test, y_test), verbose=1
    )
    histories[label] = hist.history

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for label, hist in histories.items():
    axes[0].plot(hist['loss'],     label=label, linewidth=2)
    axes[1].plot(hist['val_accuracy'], label=label, linewidth=2)

axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-entropy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_title('Validation Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print('\nFinal validation accuracy:')
for label, hist in histories.items():
    print(f'  {label:<30}: {hist["val_accuracy"][-1]:.4f}')

In [ ]:
# Measure per-layer activation variance for each config at epoch 1
def get_layer_variances(model_config, x_sample):
    label, act, init = model_config
    tf.random.set_seed(42)
    m = build_conv_net(act, init)
    m.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))
    m.fit(x_sample, np.zeros(len(x_sample), dtype=int), epochs=1, batch_size=256, verbose=0)

    # Build intermediate output model for conv and dense layers
    act_layers = [l for l in m.layers if isinstance(l, (layers.Conv2D, layers.Dense))]
    extractor  = keras.Model(inputs=m.input,
                             outputs=[l.output for l in act_layers])
    outputs    = extractor(x_sample[:256], training=False)
    return [float(tf.math.reduce_variance(tf.cast(o, tf.float32)).numpy()) for o in outputs]


variances = {}
x_sample  = x_train[:1024]

for cfg in configs:
    variances[cfg[0]] = get_layer_variances(cfg, x_sample)

fig, ax = plt.subplots(figsize=(9, 5))
for label, vars_ in variances.items():
    ax.semilogy(vars_, 'o-', label=label, linewidth=2, markersize=5)
ax.set_xlabel('Layer index (Conv/Dense only)')
ax.set_ylabel('Activation variance (log scale)')
ax.set_title('Per-Layer Activation Variance after Epoch 1')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Section 5 — `kernel_initializer` API: ResNet-Style Block

We build a residual block using the Functional API and apply targeted `kernel_initializer` choices:
- `he_normal` for all Conv layers (paired with ReLU)
- `glorot_uniform` for the projection shortcut
- `orthogonal` for a custom recurrent layer

We then inspect each layer's initialiser via `layer.kernel_initializer`.

In [ ]:
def residual_block(x, filters, stride=1):
    """Basic residual block with He init on conv layers."""
    shortcut = x

    # Main path
    x = layers.Conv2D(filters, 3, strides=stride, padding='same',
                      use_bias=False,
                      kernel_initializer='he_normal',
                      name=f'conv1_{filters}')(x)
    x = layers.BatchNormalization(name=f'bn1_{filters}')(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, 3, padding='same',
                      use_bias=False,
                      kernel_initializer='he_normal',
                      name=f'conv2_{filters}')(x)
    x = layers.BatchNormalization(name=f'bn2_{filters}')(x)

    # Projection shortcut if dimensions change
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride,
                                 use_bias=False,
                                 kernel_initializer='glorot_uniform',  # projection uses Glorot
                                 name=f'proj_{filters}')(shortcut)
        shortcut = layers.BatchNormalization(name=f'proj_bn_{filters}')(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x


# Build a small ResNet
inputs = keras.Input(shape=(32, 32, 3))
x = layers.Conv2D(16, 3, padding='same', kernel_initializer='he_normal', name='stem')(inputs)
x = layers.BatchNormalization(name='stem_bn')(x)
x = layers.ReLU()(x)
x = residual_block(x, 16)
x = residual_block(x, 32, stride=2)
x = residual_block(x, 64, stride=2)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(10, kernel_initializer='glorot_uniform', name='classifier')(x)

resnet = keras.Model(inputs, outputs, name='mini_resnet')
resnet.summary()

print('\nKernel initializer per Conv/Dense layer:')
print(f'  {"Layer name":<25} {"Type":<15} {"kernel_initializer"}')
print(f'  {"-"*60}')
for layer in resnet.layers:
    if hasattr(layer, 'kernel_initializer'):
        init_name = layer.kernel_initializer.__class__.__name__
        print(f'  {layer.name:<25} {layer.__class__.__name__:<15} {init_name}')

In [ ]:
# Add an LSTM with orthogonal recurrent_initializer
seq_inputs = keras.Input(shape=(20, 32))
lstm_out = layers.LSTM(
    64,
    kernel_initializer='glorot_uniform',
    recurrent_initializer='orthogonal',
    bias_initializer='zeros',
    name='orthogonal_lstm'
)(seq_inputs)
seq_outputs = layers.Dense(10, name='seq_out')(lstm_out)
seq_model = keras.Model(seq_inputs, seq_outputs)

lstm_layer = seq_model.get_layer('orthogonal_lstm')
print('LSTM initializer inspection:')
print(f'  kernel_initializer:     {lstm_layer.kernel_initializer.__class__.__name__}')
print(f'  recurrent_initializer:  {lstm_layer.recurrent_initializer.__class__.__name__}')
print(f'  bias_initializer:       {lstm_layer.bias_initializer.__class__.__name__}')

# Verify recurrent kernel is orthogonal (W^T W ~ I per gate block)
W_rec = lstm_layer.recurrent_kernel.numpy()  # (hidden, 4*hidden)
# Check first gate's hidden block
W_block = W_rec[:, :64]  # first gate
diff = np.abs(W_block.T @ W_block - np.eye(64)).max()
print(f'  max|W_rec_gate^T W_rec_gate - I| = {diff:.2e}  (orthogonal check)')

## Section 6 — Keras Layer Defaults Audit

We inspect the actual `kernel_initializer` and `recurrent_initializer` Keras uses by default for `Dense`, `Conv2D`, `LSTM`, and `BatchNormalization`.

In [ ]:
print('Default initializers for Keras layers:')
print(f'  {"Layer":<30} {"Attribute":<28} {"Default initializer"}')
print(f'  {"-"*80}')

# Dense
dense = layers.Dense(256)
dense.build((None, 128))
print(f'  {"Dense(256)":<30} {"kernel_initializer":<28} {dense.kernel_initializer.__class__.__name__}')
print(f'  {"":<30} {"bias_initializer":<28} {dense.bias_initializer.__class__.__name__}')

# Conv2D
conv = layers.Conv2D(64, 3)
conv.build((None, 32, 32, 3))
print(f'  {"Conv2D(64, 3)":<30} {"kernel_initializer":<28} {conv.kernel_initializer.__class__.__name__}')
print(f'  {"":<30} {"bias_initializer":<28} {conv.bias_initializer.__class__.__name__}')

# LSTM
lstm = layers.LSTM(64)
lstm.build((None, 10, 32))
print(f'  {"LSTM(64)":<30} {"kernel_initializer":<28} {lstm.kernel_initializer.__class__.__name__}')
print(f'  {"":<30} {"recurrent_initializer":<28} {lstm.recurrent_initializer.__class__.__name__}')
print(f'  {"":<30} {"bias_initializer":<28} {lstm.bias_initializer.__class__.__name__}')

# BatchNormalization
bn = layers.BatchNormalization()
bn.build((None, 32))
print(f'  {"BatchNormalization()":<30} {"gamma_initializer":<28} {bn.gamma_initializer.__class__.__name__}')
print(f'  {"":<30} {"beta_initializer":<28} {bn.beta_initializer.__class__.__name__}')
print(f'  {"":<30} {"moving_mean_initializer":<28} {bn.moving_mean_initializer.__class__.__name__}')
print(f'  {"":<30} {"moving_variance_initializer":<28} {bn.moving_variance_initializer.__class__.__name__}')

# Embedding
emb = layers.Embedding(10000, 128)
emb.build((None,))
print(f'  {"Embedding(10000, 128)":<30} {"embeddings_initializer":<28} {emb.embeddings_initializer.__class__.__name__}')

In [ ]:
# Verify default weight values
print('\nDefault weight statistics verification:')

# Dense: GlorotUniform
w = dense.kernel.numpy()
fan_in_d, fan_out_d = 128, 256
expected_var = 2.0 / (fan_in_d + fan_out_d)
print(f'  Dense kernel: var={np.var(w):.6f}  expected~{expected_var:.6f} (GlorotUniform)')

# LSTM: GlorotUniform for kernel, Orthogonal for recurrent_kernel
W_rec = lstm.recurrent_kernel.numpy()
# Check first (64,64) gate block
W_block = W_rec[:, :64]
orth_err = np.abs(W_block.T @ W_block - np.eye(64)).max()
print(f'  LSTM recurrent kernel orthogonality: max|W^T W - I| = {orth_err:.2e} (Orthogonal default)')

# BatchNorm: gamma=1, beta=0
print(f'  BatchNorm gamma (all 1): {np.allclose(bn.gamma.numpy(), 1.0)}')
print(f'  BatchNorm beta  (all 0): {np.allclose(bn.beta.numpy(),  0.0)}')
print(f'  BatchNorm moving_mean (all 0): {np.allclose(bn.moving_mean.numpy(), 0.0)}')
print(f'  BatchNorm moving_var  (all 1): {np.allclose(bn.moving_variance.numpy(), 1.0)}')

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
datasets = [
    ('Dense kernel\n(GlorotUniform)', dense.kernel.numpy().flatten()),
    ('LSTM kernel\n(GlorotUniform)',  lstm.kernel.numpy().flatten()),
    ('LSTM recurrent\n(Orthogonal)',  lstm.recurrent_kernel.numpy().flatten()),
]
for ax, (title, data) in zip(axes, datasets):
    ax.hist(data, bins=60, density=True, color='steelblue', alpha=0.8)
    ax.set_title(title); ax.set_xlabel('Weight value'); ax.set_ylabel('Density')
    ax.grid(True, alpha=0.3)

plt.suptitle('Default Keras Weight Distributions', y=1.02)
plt.tight_layout(); plt.show()